# Day 2 - Decision Tree, Random Forest and SVM (RetailMind AI)

**Goal:** try three stronger algorithms on the same data and features as Day 1, compare them fairly, and pick a champion.

**Business focus:** *recall* on the churn class. Recall = of the customers who really leave, how many did we catch? Missing a churner costs the business more than a false alarm.

## 1. Setup

In [1]:
import sys
sys.path.append("..")

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from src.config import MODEL_PATH
from src.data_loader import load_data, split_features_target
from src.pipeline import build_pipeline          # Day 1 baseline
from src.preprocessing import build_preprocessor # Day 1 preprocessing, reusable
from src.evaluate import get_metrics

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 2. Same data, same split as Day 1
Using the same `test_size`, `stratify` and `random_state` means every model is judged on the **same hold-out customers**, so the comparison is fair.

In [2]:
df = load_data()
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))

Train rows: 5634 | Test rows: 1409


## 3. Day 1 baseline (the number to beat)

In [3]:
baseline = build_pipeline(X_train)
baseline.fit(X_train, y_train)

baseline_name = "Day 1 baseline (Logistic Regression)"
results = {baseline_name: get_metrics(baseline, X_test, y_test)}
pd.DataFrame(results).T

,Accuracy,Precision,Recall,F1,ROC-AUC
Day 1 baseline (Logistic Regression),0.8055,0.6572,0.5588,0.604,0.8419


## 4. Reuse Day 1's preprocessing
`build_preprocessor()` gives each model the same cleaned and encoded features (median fill, scaling, one-hot encoding). We put it in front of every model with a `Pipeline`.

In [4]:
preprocessor = build_preprocessor(X_train)
encoded = preprocessor.fit_transform(X_train)
print("Features going into every model:", encoded.shape[1])

Features going into every model: 45


## 5. Train the three models (fixed random seed)
`random_state=42` everywhere so results are repeatable. `SVC(probability=True)` is needed so the SVM can output a churn probability for ROC-AUC.

In [5]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42),
}

for name, model in models.items():
    pipe = Pipeline([("preprocess", build_preprocessor(X_train)), ("model", model)])
    pipe.fit(X_train, y_train)
    results[name] = get_metrics(pipe, X_test, y_test)

pd.DataFrame(results).T

,Accuracy,Precision,Recall,F1,ROC-AUC
Day 1 baseline (Logistic Regression),0.8055,0.6572,0.5588,0.6040,0.8419
Decision Tree,0.7289,0.4896,0.5053,0.4974,0.6573
Random Forest,0.7779,0.6034,0.4759,0.5321,0.8162
SVM,0.7913,0.6418,0.4840,0.5518,0.7905


**Observation:** with default settings, none of the three beat the Day 1 baseline on recall. The data is imbalanced (only ~26% churn), so the models lean towards predicting "stays".

Fix: `class_weight="balanced"` tells the model that a missed churner is a bigger mistake. Let's retrain with it.

## 6. Retrain with `class_weight="balanced"`

In [6]:
balanced_models = {
    "Decision Tree (balanced)": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest (balanced)": RandomForestClassifier(class_weight="balanced", random_state=42),
    "SVM (balanced)": SVC(class_weight="balanced", probability=True, random_state=42),
}

fitted = {}
for name, model in balanced_models.items():
    pipe = Pipeline([("preprocess", build_preprocessor(X_train)), ("model", model)])
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    results[name] = get_metrics(pipe, X_test, y_test)

comparison = pd.DataFrame(results).T
comparison.sort_values("Recall", ascending=False)

,Accuracy,Precision,Recall,F1,ROC-AUC
SVM (balanced),0.7438,0.5115,0.7727,0.6155,0.8211
Day 1 baseline (Logistic Regression),0.8055,0.6572,0.5588,0.6040,0.8419
Decision Tree,0.7289,0.4896,0.5053,0.4974,0.6573
SVM,0.7913,0.6418,0.4840,0.5518,0.7905
Decision Tree (balanced),0.7282,0.4878,0.4813,0.4845,0.6496
Random Forest,0.7779,0.6034,0.4759,0.5321,0.8162
Random Forest (balanced),0.7821,0.6184,0.4679,0.5327,0.8192


**Observation**
- The **SVM** reacts strongly to `class_weight`: recall jumps a lot.
- The Decision Tree and Random Forest barely change. They are fully grown (no depth limit), so they almost memorise the training data and the class weights have little effect. That is overfitting, and **Day 3 tuning** (limiting depth) is the fix.

## 7. Pick the champion
Rule: highest **recall** among the new models, since catching at-risk customers matters most.

In [7]:
new_models = comparison.drop(index=baseline_name)
champion_name = new_models["Recall"].idxmax()
print("Champion:", champion_name)
comparison.loc[[baseline_name, champion_name]]

Champion: SVM (balanced)


,Accuracy,Precision,Recall,F1,ROC-AUC
Day 1 baseline (Logistic Regression),0.8055,0.6572,0.5588,0.6040,0.8419
SVM (balanced),0.7438,0.5115,0.7727,0.6155,0.8211


**Why it won:** the balanced SVM catches far more churners than any other model, including the baseline.

**Trade-off to be honest about:** its precision, accuracy and ROC-AUC are lower than the baseline's, so it also raises more false alarms. For a retention team that can afford to contact a few extra customers this is acceptable, but Day 3 will look for a model that keeps high recall *and* better overall quality.

## 8. Save the champion next to the Day 1 model

In [8]:
champion_path = MODEL_PATH.parent / "churn_v2.joblib"
joblib.dump(fitted[champion_name], champion_path)
print("Saved", champion_path.name, "(Day 1 model churn_pipeline.joblib is untouched)")

Saved churn_v2.joblib (Day 1 model churn_pipeline.joblib is untouched)
